# **ANN 8 – Write a Python Program to implement Adaptive Resonance Theory (ART1) for unsupervised clustering of binary input patterns.**

In [1]:
import numpy as np

In [2]:
class ART1:
    """
    An implementation of the Adaptive Resonance Theory 1 (ART1) network.
    ART1 is used for unsupervised clustering of binary input patterns.
    """
    def __init__(self, num_input, rho=0.5):
        """
        Initializes the ART1 network.

        Args:
            num_input (int): The number of features in the input vectors.
            rho (float): The vigilance parameter (0 to 1). Controls cluster granularity.
        """
        if not (0 <= rho <= 1):
            raise ValueError("Vigilance parameter rho must be between 0 and 1.")
        
        self.num_input = num_input
        self.rho = rho
        # W stores the learned cluster prototypes. It starts empty.
        self.W = np.array([])
        self.num_clusters = 0

    def train(self, X):
        """
        Trains the network by presenting a set of binary input patterns.

        Args:
            X (np.array): A 2D array of binary input patterns (samples x features).

        Returns:
            np.array: An array of cluster assignments for each input pattern.
        """
        # Initialize weights with the first pattern if the network is empty
        if self.num_clusters == 0:
            self.W = np.array([X[0]])
            self.num_clusters = 1
            assignments = [0] * len(X)
            start_index = 1
        else:
            assignments = [-1] * len(X) # -1 indicates not yet assigned
            start_index = 0

        # Process each input pattern
        for i in range(start_index, X.shape[0]):
            x = X[i]
            
            # --- Competition Phase ---
            # Calculate the net input for each existing cluster
            net_inputs = np.dot(self.W, x)
            
            # Get cluster indices sorted by similarity (highest first)
            sorted_clusters = np.argsort(net_inputs)[::-1]
            
            resonated = False
            for j in sorted_clusters:
                prototype = self.W[j]
                
                # --- Vigilance Test ---
                # Compare the input with the winning prototype
                similarity = np.sum(np.minimum(x, prototype)) / np.sum(x)
                
                if similarity >= self.rho:
                    # Resonance found: Update the prototype and assign the cluster
                    self.W[j] = np.minimum(x, prototype) # Update rule for ART1
                    assignments[i] = j
                    resonated = True
                    break
            
            if not resonated:
                # No existing cluster was a good enough match, create a new one
                self.num_clusters += 1
                self.W = np.vstack([self.W, x])
                assignments[i] = self.num_clusters - 1
        
        return np.array(assignments)

In [3]:
# --- Example Usage ---

if __name__ == "__main__":
    # Define some binary input patterns (e.g., simple shapes or letters)
    patterns = np.array([
        [1, 1, 0, 0, 0], # Pattern A
        [0, 0, 0, 1, 1], # Pattern B
        [1, 0, 0, 0, 0], # Similar to A
        [0, 0, 1, 1, 0], # New pattern C
        [0, 1, 0, 0, 0], # Distorted A
        [0, 0, 1, 1, 1]  # Similar to B
    ])

    # --- Experiment 1: High Vigilance (Strict Clustering) ---
    print("--- Experiment 1: High Vigilance (rho = 0.8) ---")
    art_high_rho = ART1(num_input=patterns.shape[1], rho=0.8)
    assignments_high = art_high_rho.train(patterns)

    print(f"Number of clusters formed: {art_high_rho.num_clusters}")
    for i, p in enumerate(patterns):
        print(f"Input Pattern {p} was assigned to Cluster {assignments_high[i]}")
    print(f"Final Prototypes (W):\n{art_high_rho.W}\n")


    # --- Experiment 2: Low Vigilance (Coarse Clustering) ---
    print("--- Experiment 2: Low Vigilance (rho = 0.4) ---")
    art_low_rho = ART1(num_input=patterns.shape[1], rho=0.4)
    assignments_low = art_low_rho.train(patterns)
    
    print(f"Number of clusters formed: {art_low_rho.num_clusters}")
    for i, p in enumerate(patterns):
        print(f"Input Pattern {p} was assigned to Cluster {assignments_low[i]}")
    print(f"Final Prototypes (W):\n{art_low_rho.W}")

--- Experiment 1: High Vigilance (rho = 0.8) ---
Number of clusters formed: 5
Input Pattern [1 1 0 0 0] was assigned to Cluster 0
Input Pattern [0 0 0 1 1] was assigned to Cluster 1
Input Pattern [1 0 0 0 0] was assigned to Cluster 0
Input Pattern [0 0 1 1 0] was assigned to Cluster 2
Input Pattern [0 1 0 0 0] was assigned to Cluster 3
Input Pattern [0 0 1 1 1] was assigned to Cluster 4
Final Prototypes (W):
[[1 0 0 0 0]
 [0 0 0 1 1]
 [0 0 1 1 0]
 [0 1 0 0 0]
 [0 0 1 1 1]]

--- Experiment 2: Low Vigilance (rho = 0.4) ---
Number of clusters formed: 4
Input Pattern [1 1 0 0 0] was assigned to Cluster 0
Input Pattern [0 0 0 1 1] was assigned to Cluster 1
Input Pattern [1 0 0 0 0] was assigned to Cluster 0
Input Pattern [0 0 1 1 0] was assigned to Cluster 1
Input Pattern [0 1 0 0 0] was assigned to Cluster 2
Input Pattern [0 0 1 1 1] was assigned to Cluster 3
Final Prototypes (W):
[[1 0 0 0 0]
 [0 0 0 1 0]
 [0 1 0 0 0]
 [0 0 1 1 1]]
